This notebook takes the output of the ortho creation step & precomputes wald

In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

2025-11-14 14:17:13.309602: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 14:17:13.312718: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

Set up the cluster

In [2]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=5,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=4)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Let's just do a super simple bounded concurrency approach

In [3]:
from pathlib import Path

In [4]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/home/mcn26/project_pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251008"

In [5]:
from dask.distributed import Semaphore, as_completed, get_client

In [6]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
#output_root=path/name/"orthos_with_precomputed_wald_erin_test_graph_mode"
#output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=2, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        return ortho_oi

In [7]:
input_ortho_names=input_ortho_names[:3]

In [8]:
import pandas as pd

In [9]:
for i in input_ortho_names:
    test_particle=precompute_one_wald(input_root=ortho_root,
        scmpradat_root=scmpradat_root,
        name=i)

    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cell_type:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cell_types.tsv",sep="\t")


    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cre:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cre.tsv",sep="\t")

    del test_particle



scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 619 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [ ]:
client.close()
cluster.close()

In [ ]:
!rm ./worker*
!rm by_cell_types.tsv
!rm by_cre.tsv

In [ ]:
#!rm -r $output_root